In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import scipy.io as sio
import h5py
import datetime
from scipy.integrate import simps
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

In [2]:
def get_grass_start_end_time(starttime_raw, endtime_raw):
    
    time_str_elements = starttime_raw.flatten()
    start_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))
    time_str_elements = endtime_raw.flatten()
    end_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))

    start_time = start_time.split(':')
    second_elements = start_time[-1].split('.')
    start_time = datetime.datetime(1990,1,1,hour=int(float(start_time[0])), minute=int(float(start_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))
    end_time = end_time.split(':')
    second_elements = end_time[-1].split('.')
    end_time = datetime.datetime(1990,1,1,hour=int(float(end_time[0])), minute=int(float(end_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))

    return start_time, end_time

def check_load_mgh_dataset(data_path, label_path, channels=None, report_and_actual_time_tol=300, reverse_sign=False):

    gender = None
    handedness = None
    
    try: # usually Grass, saved as pre 7.3 format:
        ff = sio.loadmat(signalfilepath)
        data_path = os.path.basename(signalfilepath)
        if 's' not in ff:
            raise Exception('No signal found in %s.'%data_path)
        signal = ff['s']
        if reverse_sign:
            signal = -signal
        channel_names = [ff['hdr'][0,i]['signal_labels'][0].upper().replace('EKG','ECG') for i in range(ff['hdr'].shape[1])]
        gender = None
        handedness = None
        if 'grass' in signalfilepath:
            Fs = 200.
        else:
            raise Exception('Safety check to make sure this is Grass data with Fs=200')
        # Label part for grass:
        if 'grass' in signalfilepath:
            # load labels
            with h5py.File(labelfilepath) as ffl:
                sleep_stage = ffl['stage'][()].flatten()
                apnea = ffl['apnea'][()].flatten()
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())


    except: # saved as .mat 7.3. new grass or natus files:
        with h5py.File(signalfilepath,'r') as ff:

            hdr = ff['hdr']
            signal_labels = hdr['signal_labels'][:]
            channel_names = [ ''.join(list(map(chr, ff[signal_labels[i,0]][:]))) for i in range(signal_labels.shape[0]) ]
            channel_names = [channel.upper() for channel in channel_names]
            signal = ff['s'][()]
            signal = np.transpose(signal);

            if 'recording' in ff.keys(): # only available for natus:
                year = int(np.squeeze(ff['recording']['year']))
                month = int(np.squeeze(ff['recording']['month']))
                day = int(np.squeeze(ff['recording']['day']))
                hour = int(np.squeeze(ff['recording']['hour']))
                if (hour >= 7) and (hour <=10):         # 'typo' by sleep techs
                    hour = hour + 12
                minute = int(np.squeeze(ff['recording']['minute']))
                second = int(np.squeeze(ff['recording']['second']))
                millisecond = int(np.squeeze(ff['recording']['millisecond']))
                Fs = int(np.squeeze(ff['recording']['samplingrate']))

                start_time = datetime.datetime(1990,1,1,hour=hour, minute=minute,
                        second=second, microsecond=int(millisecond*1000))
                end_time = start_time+datetime.timedelta(seconds=max(signal.shape)/Fs)

            else: # grass:
                if 'grass' in signalfilepath:
                    Fs = 200.
                else:
                    raise Exception('Safety check to make sure this is Grass data with Fs=200')

            # Label part for grass:
            ff.close()

        # load labels
        with h5py.File(labelfilepath, 'r') as ffl:
            sleep_stage = ffl['stage'][()].flatten()
            apnea = ffl['apnea'][()].flatten()
            if 'grass' in signalfilepath:
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())
            ffl.close()

    # end of loading part
    ##################################
    
    # check signal length = sleep stage length
    if sleep_stage.shape[0]!=signal.shape[1]:
        raise Exception('Inconsistent sleep stage length (%d) and signal length (%d) in %s'%(sleep_stage.shape[0],signal.shape[1],data_path))

    # only take signal channels to study
    if channels is None:
        signal_channel_ids = list(range(len(channel_names)))
        
    elif 'SumEffort' in channels:
        signal_channel_ids = []
        signal_names = []
        for ichannel in ['ABD','CHEST']:
            #channel_name_pattern = re.compile(channels[i][:2].upper()+'-*'+channels[i][-2:].upper())
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==ichannel.upper():
                    signal_channel_ids.append(j)
                    signal_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%ichannel)
        signal = signal[signal_channel_ids,:]#.T
        # do effort belt average here:
        signal = np.sum(signal,0,keepdims=1)/2
        
    else:
        signal_channel_ids = []
        signal_names = []
        for i in range(len(channels)):
            #channel_name_pattern = re.compile(channels[i][:2].upper()+'-*'+channels[i][-2:].upper())
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==channels[i].upper():
                    signal_channel_ids.append(j)
                    signal_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                continue
#                 raise Exception('Channel %s is not found.'%channels[i])
        signal = signal[signal_channel_ids,:]#.T

    # check whether the signal contains NaN
    if np.any(np.isnan(signal)):
        raise Exception('Found Nan in signal in %s'%data_path)

    # check whether sleep_stage contains all 5 stages
    stages = np.unique(sleep_stage[np.logical_not(np.isnan(sleep_stage))]).astype(int).tolist()
    if len(stages)<=1:
        if len(stages)==0:
            raise Exception('no sleep stages available in %s'%data_path)
        if stages[0] != 5: # only reasonable 1 sleepstage if all is Wake (==5), else raise Error
            raise Exception('#sleep stage <= 1: %s in %s'%(stages,data_path))

    params = {'Fs':Fs*1.0, 'signal_channel_ids':signal_channel_ids, 'signal_names':signal_names}
    if gender is not None:
        params['gender'] = gender
    if handedness is not None:
        params['handedness'] = handedness
    return signal, sleep_stage, apnea, params



In [3]:
if 1:
    
    table_grass = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/grass_studies_list.csv')
    table_natus = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/natus_studies_list.csv')
    assert np.all(table_grass.columns == table_natus.columns)
    sleeplab_table = pd.concat([table_grass, table_natus], axis=0)
    sleeplab_table = sleeplab_table[pd.notna(sleeplab_table.MRN)]
    sleeplab_table.reset_index(inplace=True, drop=True)
    
    sleeplab_table['MRN_int'] = [int(str(x).replace('-','').replace('/','').replace('X','0')) for x in sleeplab_table['MRN']]
    dates = pd.to_datetime(sleeplab_table.DateOfVisit, infer_datetime_format=1)


    for jloc, row in sleeplab_table.iterrows():
        path = row.Path
        path = path.replace('M:', '/media/mad3')
        path = path.replace('\\', '/')

        signalfile = [x for x in os.listdir(path) if 'Signal' in x]
        if len(signalfile) > 0:
            signalfile = signalfile[0]
        else:
            continue 

        labelfile = [x for x in os.listdir(path) if 'Labels' in x]
        if len(labelfile) > 0:
            labelfile = labelfile[0]
        else:
            continue

        sleeplab_table.loc[jloc, 'Signalfile'] = os.path.join(path, signalfile)
        sleeplab_table.loc[jloc, 'Labelfile'] = os.path.join(path, labelfile)

    sleeplab_table.to_csv('sleeplab_table.csv', index=False)

else:
    sleeplab_table = pd.read_csv('sleeplab_table.csv')

In [4]:
hypoxic_table = pd.read_excel('Hypoxic_Study_Table.xlsx', engine='openpyxl')

control_table = pd.read_excel('/scr/wolfgang/Hypoxia_Study/control_group.xls')

In [5]:
control_table = control_table[control_table.MRN_Type.apply(lambda x: 'MGH' in x)]
control_table['MRN_original'] = control_table['MRN']
control_table.loc[control_table.MRN_Type == 'PMRN, MGH, BWH', 'MRN'] = control_table.loc[control_table.MRN_Type == 'PMRN, MGH, BWH'].MRN.apply(lambda x: x.split(',')[1].strip())
control_table.loc[control_table.MRN_Type == 'PMRN, MGH', 'MRN'] = control_table.loc[control_table.MRN_Type == 'PMRN, MGH'].MRN.apply(lambda x: x.split(',')[1].strip())
control_table['MRN'] = control_table['MRN'].astype(int)

In [6]:
# print('TMP!!!!!!')

# hypoxic_table = hypoxic_table.iloc[:20]
# control_table = control_table.iloc[:20]


In [7]:
def mrn_matching(table):
    # first, let's check if there is a perfect match with MRN for MGH patients.
    table['MRN_Match'] = 0
    
#     table_MGH = table[table.MRN_Type == 'MGH']
    table.loc[table.index,'MRN_Match'] = np.isin(table.MRN, sleeplab_table.MRN_int)
    
    if 'Name' in table.columns:
        table['LastName'] = [str(x).split(', ')[0] for x in table.Name.values]
        table['FirstName'] = [str(x).split(', ')[1] if ', ' in str(x) else np.nan for x in table.Name.values]

    return table

hypoxic_table = mrn_matching(hypoxic_table)
control_table = mrn_matching(control_table)

In [8]:
import Levenshtein
import pdb; 

In [9]:
if 1:
    hypoxic_table['NameMatch'] = np.nan
    hypoxic_table['NewMRN'] = np.nan

    for index, row in hypoxic_table.iterrows():

        for index2, row2 in sleeplab_table.iterrows():
            try:
                if Levenshtein.distance(str(row.LastName), str(row2.LastName)) <= 2:
                    if Levenshtein.distance(str(row.FirstName), str(row2.FirstName)) <= 2:
                        hypoxic_table.loc[index, 'NameMatch'] = 1
                        hypoxic_table.loc[index, 'MatchedMRN'] = int(row2.MRN_int)
                        hypoxic_table.loc[index, 'MatchedLastName'] = row2.LastName
                        hypoxic_table.loc[index, 'MatchedFirstName'] = row2.FirstName
                        hypoxic_table.loc[index, 'MatchedDOB'] = row2.DateOfBirth

            except Exception as e:
                if not str(e) == 'distance expected two Strings or two Unicodes':
                    print(e)
                continue


In [10]:
hypoxic_table.MRN_Match.sum()

409

In [11]:
print(hypoxic_table.MRN_Match.sum())
print(control_table.MRN_Match.sum())


409
8199


In [12]:
from sleep_analysis_functions import compute_spo2_clean, compute_hypoxia_burden, desaturation_detection, hypoxaemic_burden_minutes

def compute_hypoxia_statistics(signalfilepath, labelfilepath):
    
    signal, sleep_stage, apnea, params = check_load_mgh_dataset(signalfilepath, labelfilepath, channels=['SAO2','SPO2','PTAF', 'CFlow'], report_and_actual_time_tol=300, reverse_sign=False)
    
    data = pd.DataFrame()
    data['Apnea'] = apnea
    for iSignal, name in enumerate(params['signal_names']):
        data[name] = signal[iSignal]
    data.rename({'SAO2':'SPO2'}, axis=1, inplace=True)
    data['spo2'] = data['SPO2']
    fs = int(params['Fs'])

    dt_start = pd.Timestamp('2000-01-01 00:00:00')
    dt_end = dt_start + datetime.timedelta(seconds=(data.shape[0]-1) / fs)
    pseudo_dt_index = pd.date_range(start=dt_start, end=dt_end, periods=data.shape[0])
    data.index = pseudo_dt_index

    data = compute_spo2_clean(data, fs=fs)
    data['spo2'] = data['spo2_clean']

    data['Apnea_Binary'] = np.isin(data['Apnea'],[1,2,3,4]).astype(int)
    data['Apnea_End'] = np.isin(data['Apnea_Binary'].diff(), [-1])
    
    sleep_stage = sleep_stage[np.logical_not(pd.isna(sleep_stage))]
    hours_sleep = sum(sleep_stage<5)/fs/3600
    
    data, hypoxia_burden = compute_hypoxia_burden(data, fs, hours_sleep=hours_sleep)
    
    
    T90burden, T90desaturation, T90nonspecific = hypoxaemic_burden_minutes(data['spo2'].values, fs)
    
    AHI = np.round(sum(data['Apnea_End'])/hours_sleep,1)

    return hypoxia_burden, AHI, hours_sleep, T90burden, T90desaturation, T90nonspecific


In [13]:
sleeplab_avail_full = pd.DataFrame()

# hypoxic_table[f'Visit_1_HypoxiaBurden'] = np.nan
for i_visit in range(10):
    hypoxic_table.loc[index,f'Visit_{i_visit+1}_HypoxiaBurden'] = np.nan
    hypoxic_table.loc[index,f'Visit_{i_visit+1}_AHI'] = np.nan
    hypoxic_table.loc[index,f'Visit_{i_visit+1}_Total_Sleep_Time'] = np.nan
    hypoxic_table.loc[index,f'Visit_{i_visit+1}_T90_minutes'] = np.nan
    hypoxic_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_desat'] = np.nan
    hypoxic_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_unspecific'] = np.nan

for index, row in tqdm(hypoxic_table.iterrows()):
#     pdb.set_trace()
    try:
        new_patient = 0

        if row['MRN_Match'] == 1:
            sleeplab_avail = sleeplab_table[np.isin(sleeplab_table.MRN_int, row.MRN)].copy()
        elif row['NameMatch'] == 1:
            sleeplab_avail = sleeplab_table[np.isin(sleeplab_table.MRN_int, row.MatchedMRN)].copy()
        else:
            continue

        num_of_visits = sleeplab_avail.shape[0]
        hypoxic_table.loc[index,'num_of_visits'] = num_of_visits

        i_visit = -1
        for index_sleeplab, row_sleeplab in sleeplab_avail.iterrows():

            i_visit += 1

    #         if not f'Visit_{i_visit+1}_HypoxiaBurden' in hypoxic_table.columns:
    #             do_nothing = 1
            if not pd.isna(hypoxic_table.loc[index,f'Visit_{i_visit+1}_HypoxiaBurden']):
                # this has already been computed in prior loop/computation, let's skip it.
                continue

            # if there is at least one visist which is new, do computation and add to table eventually:
            new_patient = 1

            hypoxic_table.loc[index,f'Visit_{i_visit+1}_Date'] = sleeplab_avail.iloc[i_visit].DateOfVisit		
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_Type'] = sleeplab_avail.iloc[i_visit].TypeOfTest

            path = sleeplab_avail.iloc[i_visit].Path
            signalfile = sleeplab_avail.iloc[i_visit].Signalfile
            labelfile = sleeplab_avail.iloc[i_visit].Labelfile

            hypoxic_table.loc[index,f'Visit_{i_visit+1}_Path'] = path
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_Signalfile'] = signalfile
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_Labelfile'] = labelfile

            signalfilepath = signalfile # os.path.join(path, signalfile)
            labelfilepath = labelfile # os.path.join(path, labelfile)

            if not pd.isna(labelfile): #  == '[]':
    #             try:
                hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signalfilepath, labelfilepath)

    #             except Exception as e:
    #                 print(e)
    #                 print(signalfilepath)
    #                 print(labelfilepath)
    #                 continue

            else:
                [hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific] = [np.nan]*6

            hypoxic_table.loc[index,f'Visit_{i_visit+1}_HypoxiaBurden'] = [hypoxia_burden]
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_AHI'] = [ahi]
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_Total_Sleep_Time'] = [total_sleep_time]
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_T90_minutes'] = [T90burden]
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_desat'] = [T90desaturation]
            hypoxic_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_unspecific'] = [T90nonspecific]

            # also fill up sleeplab_available table:
            sleeplab_avail.loc[index_sleeplab, 'Hypoxia_Burden'] = [hypoxia_burden]
            sleeplab_avail.loc[index_sleeplab, 'AHI'] = [ahi]
            sleeplab_avail.loc[index_sleeplab, 'Total_Sleep_Time'] = [np.round(total_sleep_time,1)]
            sleeplab_avail.loc[index_sleeplab, 'T90_minutes'] = [np.round(T90burden,1)]
            sleeplab_avail.loc[index_sleeplab, 'T90_minutes_desat'] = [np.round(T90desaturation,1)]
            sleeplab_avail.loc[index_sleeplab, 'T90_minutes_unspecific'] = [np.round(T90nonspecific,1)]

        if new_patient == 1:
            sleeplab_avail_full = pd.concat([sleeplab_avail_full, sleeplab_avail], axis=0)
            sleeplab_avail_full.to_csv('sleeplab_avail_full_tmp.csv', index=False)
            hypoxic_table.to_csv('hypoxic_table_results_tmp.csv', index=False)

    except Exception as e:
        print(e)
        continue

0it [00:00, ?it/s]/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:490: RuntimeWarning: invalid value encountered in greater_equal
  desaturation_spo2[spo2_signal >= minimum_spo2_level] = 0
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/lib/nanfunctions.py:1391: RuntimeWarning: All-NaN slice encountered
  overwrite_input, interpolation)
21it [05:04, 17.51s/it]/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:464: RuntimeWarning: invalid value encountered in less
  spo2_drop[spo2_drop<0] = 0
/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:478: RuntimeWarning: invalid value encountered in less
  areas_robust = areas[areas < np.std(areas)*3]
28it [08:55, 55.51s/it]

Timestamp('2000-01-01 03:13:36.846875188')


37it [13:56, 25.05s/it]

index 10123270 is out of bounds for axis 0 with size 10123270


47it [20:09, 58.35s/it]

...

890it [7:16:31, 47.45s/it]

Timestamp('2000-01-01 01:46:38.007031139')


919it [7:27:57, 59.21s/it]

Timestamp('2000-01-01 00:40:27.698046917')


936it [7:31:20, 28.93s/it]


In [14]:
hypoxic_table.dropna(how='all', axis=1, inplace=True)

In [15]:
hypoxic_table.loc[pd.notna(hypoxic_table['Visit_1_HypoxiaBurden'])].iloc[:, 40:55]

,LVsat,MRN_Match,LastName,FirstName,NameMatch,MatchedMRN,MatchedLastName,MatchedFirstName,MatchedDOB,Visit_1_HypoxiaBurden,Visit_1_AHI,Visit_1_Total_Sleep_Time,Visit_1_T90_minutes,Visit_1_T90_minutes_desat,Visit_1_T90_minutes_unspecific
2,NaN,True,Matthews,John,1.0,978221.0,Matthews,John D,1955-01-04,14.3,5.9,7.761111,4.449417,4.149667,0.299750
10,NaN,False,Correnti,Karen,1.0,5643580.0,Corsetti,Aaron,1957-09-14,41.6,32.8,6.767500,1.477417,1.477417,0.000000
14,NaN,True,Frishkopf,Lawrence S,1.0,836334.0,Frishkopf,Lawrence S,1930-06-26,30.7,12.2,6.866667,8.312917,8.312917,0.000000
18,NaN,True,Fitzgerald,Richard P,1.0,1996747.0,Fitzgerald,Richard P,1938-01-21,1.9,2.8,7.166667,0.565833,0.188667,0.377167
20,NaN,True,Lascuola,John,NaN,NaN,NaN,NaN,NaN,0.6,1.2,5.827222,3.400167,2.878000,0.522167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
907,NaN,True,WEBB,JEFFREY,NaN,NaN,NaN,NaN,NaN,0.0,0.1,7.524993,0.000000,0.000000,0.000000
911,NaN,False,Gardner,Ian D,1.0,574659.0,Gardiner,Ian H,1946-09-13,2.3,1.8,6.600000,0.000000,0.000000,0.000000
912,NaN,True,Wise,John,1.0,1493518.0,West,John,1961-07-20,2.6,4.3,6.908311,1.299674,1.299674,0.000000
920,NaN,False,Garcia,Milagros,1.0,3001358.0,Garcia,Milagros,1960-04-28,0.0,0.5,4.350000,0.000000,0.000000,0.000000


In [24]:
hypoxic_table.loc[hypoxic_table.LastName == 'Roderick'].iloc[:, 45 : 60]

,MatchedMRN,MatchedLastName,MatchedFirstName,MatchedDOB,Visit_1_HypoxiaBurden,Visit_1_AHI,Visit_1_Total_Sleep_Time,Visit_1_T90_minutes,Visit_1_T90_minutes_desat,Visit_1_T90_minutes_unspecific,Visit_2_HypoxiaBurden,Visit_2_AHI,Visit_2_Total_Sleep_Time,Visit_2_T90_minutes,Visit_2_T90_minutes_desat
284,4738703.0,Roderick,Donald W,1946-04-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
control_table.shape

(18633, 18)

In [17]:
sleeplab_avail_full = pd.DataFrame()

for i_visit in range(10):
    control_table.loc[index,f'Visit_{i_visit+1}_HypoxiaBurden'] = np.nan
    control_table.loc[index,f'Visit_{i_visit+1}_AHI'] = np.nan
    control_table.loc[index,f'Visit_{i_visit+1}_Total_Sleep_Time'] = np.nan
    control_table.loc[index,f'Visit_{i_visit+1}_T90_minutes'] = np.nan
    control_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_desat'] = np.nan
    control_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_unspecific'] = np.nan

for index, row in tqdm(control_table.iterrows()):

    try:
        new_patient = 0

        if row['MRN_Match'] == 1:
            sleeplab_avail = sleeplab_table[np.isin(sleeplab_table.MRN_int, row.MRN)].copy()
        else:
            continue

        num_of_visits = sleeplab_avail.shape[0]
        control_table.loc[index,'num_of_visits'] = num_of_visits

        i_visit = -1
        for index_sleeplab, row_sleeplab in sleeplab_avail.iterrows():

            i_visit += 1

    #         if not f'Visit_{i_visit+1}_HypoxiaBurden' in control_table.columns:
    #             do_nothing = 1
            if not pd.isna(control_table.loc[index,f'Visit_{i_visit+1}_HypoxiaBurden']):
                # this has already been computed in prior loop/computation, let's skip it.
                continue

            # if there is at least one visist which is new, do computation and add to table eventually:
            new_patient = 1
            control_table.loc[index,f'Visit_{i_visit+1}_Date'] = sleeplab_avail.iloc[i_visit].DateOfVisit		
            control_table.loc[index,f'Visit_{i_visit+1}_Type'] = sleeplab_avail.iloc[i_visit].TypeOfTest

            path = sleeplab_avail.iloc[i_visit].Path
            signalfile = sleeplab_avail.iloc[i_visit].Signalfile
            labelfile = sleeplab_avail.iloc[i_visit].Labelfile

            control_table.loc[index,f'Visit_{i_visit+1}_Path'] = path
            control_table.loc[index,f'Visit_{i_visit+1}_Signalfile'] = signalfile
            control_table.loc[index,f'Visit_{i_visit+1}_Labelfile'] = labelfile

            signalfilepath = signalfile # os.path.join(path, signalfile)
            labelfilepath = labelfile # os.path.join(path, labelfile)

            if not pd.isna(labelfile): #  == '[]':
    #             try:
                hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signalfilepath, labelfilepath)

    #             except Exception as e:
    #                 print(e)
    #                 print(signalfilepath)
    #                 print(labelfilepath)
    #                 continue

            else:
                [hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific] = [np.nan]*6

            control_table.loc[index,f'Visit_{i_visit+1}_HypoxiaBurden'] = [hypoxia_burden]
            control_table.loc[index,f'Visit_{i_visit+1}_AHI'] = [ahi]
            control_table.loc[index,f'Visit_{i_visit+1}_Total_Sleep_Time'] = [total_sleep_time]
            control_table.loc[index,f'Visit_{i_visit+1}_T90_minutes'] = [T90burden]
            control_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_desat'] = [T90desaturation]
            control_table.loc[index,f'Visit_{i_visit+1}_T90_minutes_unspecific'] = [T90nonspecific]

            # also fill up sleeplab_available table:
            sleeplab_avail.loc[index_sleeplab, 'Hypoxia_Burden'] = [hypoxia_burden]
            sleeplab_avail.loc[index_sleeplab, 'AHI'] = [ahi]
            sleeplab_avail.loc[index_sleeplab, 'Total_Sleep_Time'] = [np.round(total_sleep_time,1)]
            sleeplab_avail.loc[index_sleeplab, 'T90_minutes'] = [np.round(T90burden,1)]
            sleeplab_avail.loc[index_sleeplab, 'T90_minutes_desat'] = [np.round(T90desaturation,1)]
            sleeplab_avail.loc[index_sleeplab, 'T90_minutes_unspecific'] = [np.round(T90nonspecific,1)]

        if new_patient == 1:
            sleeplab_avail_full = pd.concat([sleeplab_avail_full, sleeplab_avail], axis=0)
            sleeplab_avail_full.to_csv('sleeplab_avail_full_tmp.csv', index=False)
            control_table.to_csv('control_table_results_tmp.csv', index=False)

    except Exception as e:
        print(e)
        continue

31it [07:15, 44.08s/it]

Timestamp('2000-01-01 02:29:38.939843795')


38it [09:40, 33.91s/it]

local variable 'samples_limit' referenced before assignment


52it [11:56, 32.75s/it]

...

9897it [56:04:05, 97.23s/it]

Timestamp('2000-01-01 00:33:04.383203167')


9939it [56:26:54, 68.30s/it]

local variable 'samples_limit' referenced before assignment


KeyboardInterrupt: 

In [ ]:
control_table.to_csv('hypoxia_control_table.csv')
sleeplab_avail_full.to_csv('sleeplab_table_with_hypoxia.csv')

In [ ]:
control_table.loc[pd.notna(control_table['Visit_1_HypoxiaBurden'])].iloc[:, :25]

In [ ]:
sleeplab_avail_full.columns

In [ ]:
sleeplab_avail_share = sleeplab_avail_full.drop(['Signalfile', 'System','Labelfile','FolderName','FolderPath','PatientID', 'Path'], axis=1)

In [ ]:
sleeplab_avail_share = sleeplab_avail_share[['FirstName','LastName','DateOfBirth', 'Sex', 'MRN', 'MRN_int', 'DateOfVisit','TypeOfTest', 'Total_Sleep_Time', 'AHI', 'Hypoxia_Burden']]

In [ ]:
sleeplab_avail_share.to_csv('psg_studies_ahi_hypoxia_burden.csv', index=False)

In [ ]:
hypoxic_table.to_csv('hypoxic_table_with_psg_data.csv', index=False)

In [27]:
sleeplab_avail_full.shape

(712, 18)

In [43]:
hypoxic_table.columns[:10]

Index(['File', 'Line', 'ID', 'Name', 'Multiple values', 'EMPI', 'EPIC_PMRN',
       'MRN_Type', 'MRN', 'Report_Number'],
      dtype='object')

In [93]:
selection = (hypoxic_table['MRN_Match']==0) & (hypoxic_table['NameMatch']==1)
sel_table = hypoxic_table.loc[selection, ['Name', 'MRN_Type', 'MRN', 'NameMatch','MatchedMRN', 'MatchedLastName', 'MatchedFirstName']].copy()
sel_table.reset_index(inplace=True)
sel_table.drop(['index'],axis=1, inplace=True)
sel_table

,Name,MRN_Type,MRN,NameMatch,MatchedMRN,MatchedLastName,MatchedFirstName
0,"Waite, Suzanne",BWH,14048110,1.0,1824261.0,Waite,Suzanne
1,"Giroux, Margarete",BWH,24416349,1.0,4373783.0,Giroux,Margarete
2,"Swain, Richard",BWH,31848708,1.0,5905412.0,Swain,Richard
3,"Carey, William",MGH,1599843,1.0,2274159.0,Carey,William H
4,"Walsh, David P",BWH,11924768,1.0,3167268.0,Walsh,David L
5,"Deberardinis, Mario",MGH,2468714,1.0,1628027.0,Deberardinis,Marion
6,"Washington, Susan",BWH,2134559,1.0,2196761.0,Washington,Susan
7,"Noiles, Nina",BWH,5697594,1.0,5308280.0,Noiles,Nina
8,"Stapleton, Tammy A",BWH,31549488,1.0,5655499.0,Stapleton,Tammy
9,"Doiron, Mary",BWH,14044580,1.0,5158589.0,Doiron,Mary V


In [94]:
sel_table.to_csv('matched_based_on_name.csv')

In [54]:
hypoxic_table.drop(['NewMRN'], axis=1, inplace=True)

In [66]:
hypoxic_table[['Name','Visit_1_Date','Visit_1_HypoxiaBurden', 'Visit_1_AHI', 'Visit_1_Total_Sleep_Time']].dropna(subset=['Visit_1_Date'])

,Name,Visit_1_Date,Visit_1_HypoxiaBurden,Visit_1_AHI,Visit_1_Total_Sleep_Time
2,"Matthews, John",08/14/2009,13.7,6.1,7.761111
14,"Frishkopf, Lawrence S",08/11/2017,30.7,12.2,6.866667
18,"Fitzgerald, Richard P",10/18/2017,2.5,2.8,7.166667
20,"Lascuola, John",04/19/2014,0.6,1.2,5.827222
24,"Bernardi, Richard J",05/06/2016,112.5,37.2,5.356389
...,...,...,...,...,...
912,"Wise, John",12/04/2011,26.6,19.3,5.713333
913,"Boucher, Jessica",10/20/2010,0.0,0.0,3.652778
918,"Dowling, Carol",12/19/2014,6.4,7.3,6.441667
920,"Garcia, Milagros",01/05/2010,0.0,0.5,4.350000


In [61]:
hypoxic_table.columns

Index(['File', 'Line', 'ID', 'Name', 'Multiple values', 'EMPI', 'EPIC_PMRN',
       'MRN_Type', 'MRN', 'Report_Number',
       ...
       'Visit_10_AHI', 'Visit_10_Total_Sleep_Time', 'Visit_11_Date',
       'Visit_11_Type', 'Visit_11_Path', 'Visit_11_Signalfile',
       'Visit_11_Labelfile', 'Visit_11_HypoxiaBurden', 'Visit_11_AHI',
       'Visit_11_Total_Sleep_Time'],
      dtype='object', length=143)

In [28]:
sleeplab_avail_full.to_csv('sleeplab_avail_full.csv', index=False)
# sleeplab_avail_full

In [29]:
hypoxic_table.to_csv('hypoxic_table_results.csv', index=False)


In [ ]:
sleeplab_avail[['LastName', 'FirstName', 'DateOfBirth', 'TypeOfTest']]

In [32]:
sleeplab_avail = sleeplab_table[np.isin(sleeplab_table.MRN_int, hypoxic_table_MGH.MRN)]

In [33]:
hypoxic_avail[['Name','MRN','Report_Date_Time']]

,Name,MRN,Report_Date_Time
2,"Matthews, John",978221,2010-03-29 01:32:00
14,"Frishkopf, Lawrence S",836334,2018-08-17 18:13:00
18,"Fitzgerald, Richard P",1996747,2018-06-13 17:25:00
20,"Lascuola, John",1558684,2009-04-07 11:35:00
24,"Bernardi, Richard J",2485662,2016-04-11 14:00:00
...,...,...,...
904,"Graf, Cynthia",5799198,2017-01-18 13:39:00
907,"WEBB, JEFFREY",3690069,2014-02-27 08:47:00
909,"Hazimeh, Khalid",3283155,2020-01-30 09:24:00
912,"Wise, John",1262196,2019-03-05 20:02:00


In [34]:
sleeplab_avail[sleeplab_avail.MRN_int==978221]

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Signalfile,Labelfile,System,Path,FolderPath,MRN_int
14178,Matthews_John_081409_2202.000,Z6587649,097-82-21,Matthews,John D,Male,1/4/1955,08/14/2009,PSG all night CPAP,Signal_Matthews_John_081409_2202.000.mat,Labels_Matthews_John_081409_2202.000.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,978221
16207,Matthews_John_081812_2255.000,Z6587649,097-82-21,Matthews,John D,Male,1/4/1955,08/18/2012,PSG all night CPAP,Signal_TwinData6_686.mat,Labels_TwinData6_686.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,978221
17693,Matthews_John_072409_2308.000,Z6587649,097-82-21,Matthews,John D,Male,1/4/1955,07/24/2009,PSG Split night,Signal_Matthews_John_072409_2308.000.mat,Labels_Matthews_John_072409_2308.000.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,978221


In [38]:
sleeplab_avail

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Signalfile,Labelfile,System,Path,FolderPath,MRN_int
15,Deberardinis_Marion_072817_2242.000,Z6377495,162-80-27,Deberardinis,Marion,Female,12/27/1950,07/28/2017,PSG Diagnostic,Signal_Deberardinis_Marion_072817_2242.000.mat,Labels_Deberardinis_Marion_072817_2242.000.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,1628027
74,Ansaldi_Laurie_121412_2300.000,Z11636199,225-96-68,Ansaldi,Laurie,Female,5/30/1961,12/14/2012,PSG all night CPAP,Signal_TwinData6_Exported_30.mat,Labels_TwinData6_30.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,2259668
149,Amazu_Adeline_050714_2348.000,Z12907166,345-73-99,Amazu,Adeline,Female,9/4/1949,05/07/2014,PSG all night CPAP,Signal_TwinData10_Exported_14.mat,Labels_TwinData10_14.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,3457399
179,El-Sherbini_Ahmad_060209_2306.000,Z6365231,250-52-26,Elsherbini,Ahmed A,Male,9/1/1940,06/02/2009,PSG Diagnostic,Signal_El-Sherbini_Ahmad_060209_2306.000.mat,Labels_El-Sherbini_Ahmad_060209_2306.000.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,2505226
215,Mullen_Robert_110211_2156.000,Z8714448,265-72-98,Mullen,Robert K,Male,5/25/1931,11/02/2011,PSG all night CPAP,Signal_Mullen_Robert_110211_2156.000.mat,Labels_Mullen_Robert_110211_2156.000.mat,Grass,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,2657298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24164,Vogel~ Michael_da9c24b2-7cde-4d3d-b60d-d68cde0...,Z8106184,100-09-32,Vogel,Michael H,Male,1951-02-01,2019-04-16,Split Night,[],[],Natus,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,1000932
24223,Marrero~ Hecto_3d1420ec-7c87-41dc-8026-d5968c8...,Z12068005,348-06-30,Marrero,Hector M,Male,1968-08-17,2018-08-20,Diagnostic,Signal_Marrero~ Hecto_3d1420ec-7c87-41dc-8026-...,Labels_Marrero~ Hecto_3d1420ec-7c87-41dc-8026-...,Natus,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,3480630
24334,Palermo~ Julie_af85aaa1-7868-43b5-909a-954794f...,Z9014512,265-73-18,Palermo,Julie A,Female,1957-08-02,2018-08-26,Diagnostic,Signal_Palermo~ Julie_af85aaa1-7868-43b5-909a-...,Labels_Palermo~ Julie_af85aaa1-7868-43b5-909a-...,Natus,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,2657318
24338,Leon~ Victor_0b69fab1-6753-4b08-89dc-7e042cd3446c,Z9144001,280-83-92,Leon,Victor J.,Male,1949-08-27,2018-12-11,Split Night,Signal_Leon~ Victor_0b69fab1-6753-4b08-89dc-7e...,Labels_Leon~ Victor_0b69fab1-6753-4b08-89dc-7e...,Natus,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,2808392


In [308]:
study = sleeplab_avail.iloc[2]
study

FolderName                         Amazu_Adeline_050714_2348.000
PatientID                                              Z12907166
MRN                                                    345-73-99
LastName                                                   Amazu
FirstName                                                Adeline
Sex                                                       Female
DateOfBirth                                             9/4/1949
DateOfVisit                                           05/07/2014
TypeOfTest                                    PSG all night CPAP
Signalfile                     Signal_TwinData10_Exported_14.mat
Labelfile                               Labels_TwinData10_14.mat
System                                                     Grass
Path           /media/mad3/Datasets_ConvertedData/sleeplab/gr...
FolderPath                                                   NaN
MRN_int                                                  3457399
Name: 149, dtype: object

In [309]:
signalfilepath = os.path.join(study.Path, study.Signalfile)
labelfilepath = os.path.join(study.Path, study.Labelfile)

In [310]:
signalfilepath

'/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Amazu_Adeline_050714_2348.000/Signal_TwinData10_Exported_14.mat'

In [326]:
x = np.arange(-50,50+1/fs,1/fs)
plt.figure()
plt.plot(x,mean_spo2)
plt.scatter(x[search_window0],mean_spo2[search_window0],c='r',zorder=5)
plt.scatter(x[search_window1],mean_spo2[search_window1],c='r', zorder=5)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [348]:
plt.figure()
plt.plot(np.arange(len(spo2_drop))/fs,spo2_drop)
plt.title(area_tmp)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [357]:
# def hypoxia_plot():

plt.rcParams.update({'font.size': 8})
seconds_roi = 10
flow_seconds_drop = 15
fontdict = {'fontsize':8}

fig = plt.figure()

ax1 = fig.add_subplot(411)
# plt.plot(data.Stage,'r')
ax1.plot(data.Apnea)
ax1.set_yticks(np.arange(7))
ax1.set_yticklabels(['No Event', 'OA','CA','MA','HY','','RERA'])
ax1.set_ylabel('Apnea Label', fontdict=fontdict)

if 'PTAF' in data.columns:
    ax2 = fig.add_subplot(412, sharex=ax1)
    ax2.plot(data.PTAF)
    ax2.scatter(data.index[data['Apnea_End']], data.PTAF[data['Apnea_End']], color='r',s=10, zorder=5)
    
    # ax2.scatter(cpap_on_dt, data.PTAF.loc[cpap_on_dt], c='r', zorder=10)
    ax2.set_ylabel('PTAF', fontdict=fontdict)

if 'CFLOW' in data.columns:
    ax3 = fig.add_subplot(413, sharex=ax1)
    ax3.plot(data.CFLOW)
    ax3.scatter(data.index[data['Apnea_End']], data.CFLOW[data['Apnea_End']], color='r',s=10, zorder=5)
    ax3.set_ylabel('CFlow', fontdict=fontdict)

ax4 = fig.add_subplot(414, sharex=ax1)
ax4.plot(data.SPO2)
ax4.scatter(data.index[data['Apnea_End']], data.SPO2[data['Apnea_End']], color='r',s=10, zorder=5)
ax4.scatter(idx_prior, data.SPO2[idx_prior], color='g',s=10, zorder=5)
ax4.scatter(idx_post, data.SPO2[idx_post], color='cyan',s=12, zorder=5)
ax4.scatter(data.index[data['Apnea_End']], spo2_refs, color='purple', s=10, zorder=5)
            
            
ax4.set_ylabel('SpO2', fontdict=fontdict)
ax4.set_ylim([data.SPO2.quantile(0.01), data.SPO2.max()+0.5])
ax4.set_xlabel('time (s)')

# ax.set_xlim([pd.Timestamp('2019-01-25 00:17:00.00'), pd.Timestamp('2019-01-25 00:21:00.00')])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 0, 'time (s)')

In [297]:
data.SPO2[np.round(idx_post,1)]

4753.3     94.555581
5186.6     93.565270
11894.8    89.604028
11966.4    87.624933
19985.1    87.624933
20055.5    88.613718
Name: SPO2, dtype: float64

In [300]:
for i in range(25):
    plt.close()

In [299]:
idx_post

Float64Index([4753.3, 5186.6, 11894.8, 11966.4, 19985.1, 20055.5], dtype='float64')

In [13]:
signalfilepath = '/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Anatasia_Lori_091611_2209.000/Signal_TwinData5_Exported_34.mat'